In [ ]:
# CG-HAF: Clinically-Guided XAI for Dermatology with Baseline Comparisons
# FINAL CORRECTED VERSION - Fixes Asymmetry, Pink Hue, and Syntax Errors

import os
import gc
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.models import swin_v2_b
import cv2
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

# Memory management
def clear_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

# Create output directory
output_dir = "cghaf_results_final"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, "xai_visualizations"), exist_ok=True)

# ========================================
# MODEL ARCHITECTURE
# ========================================
class EnhancedSwinTransformer(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.2, use_batchnorm=False):
        super(EnhancedSwinTransformer, self).__init__()
        self.backbone = swin_v2_b(weights=None)
        num_features = self.backbone.head.in_features
        self.backbone.head = nn.Identity()
        
        if use_batchnorm:
            self.classifier = nn.Sequential(
                nn.Dropout(dropout_rate),
                nn.Linear(num_features, num_features // 4),
                nn.ReLU(),
                nn.BatchNorm1d(num_features // 4),
                nn.Dropout(dropout_rate / 2),
                nn.Linear(num_features // 4, num_classes)
            )
        else:
            self.classifier = nn.Sequential(
                nn.Dropout(dropout_rate),
                nn.Linear(num_features, num_features // 4),
                nn.ReLU(),
                nn.LayerNorm(num_features // 4),
                nn.Dropout(dropout_rate / 2),
                nn.Linear(num_features // 4, num_classes)
            )

    def forward(self, x):
        if self.training and x.size(0) == 1:
            x = x.repeat(2, 1, 1, 1)
            features = self.backbone(x)
            return self.classifier(features)[:1]
        features = self.backbone(x)
        return self.classifier(features)

# ========================================
# DATASET
# ========================================
class HAM10000Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'path']
        label = self.df.loc[idx, 'label']
        try:
            with Image.open(img_path) as img:
                image = img.convert("RGB").copy()
        except:
            image = Image.new('RGB', (224, 224), color='black')
        if self.transform:
            image = self.transform(image)
        return image, label, img_path

# ========================================
# DATA LOADING
# ========================================
def load_test_data(metadata_path, img_dir1, img_dir2):
    df = pd.read_csv(metadata_path)
    image_paths = {}
    for img_name in tqdm(df['image_id'], desc="Mapping paths"):
        path1 = os.path.join(img_dir1, img_name + ".jpg")
        path2 = os.path.join(img_dir2, img_name + ".jpg")
        if os.path.exists(path1):
            image_paths[img_name] = path1
        elif os.path.exists(path2):
            image_paths[img_name] = path2
    df = df[df['image_id'].isin(image_paths.keys())]
    df['path'] = df['image_id'].map(image_paths)
    return df

# ========================================
# INNOVATION 1: CLINICAL SALIENCY MASK (ABCD Rule) - FIXED
# ========================================
class ClinicalSaliencyMask:
    def __init__(self, w_A=1.0, w_B=1.2, w_C=0.9, w_D=1.0):
        self.w_A, self.w_B, self.w_C, self.w_D = w_A, w_B, w_C, w_D

    def compute_asymmetry(self, image):
        """FIXED: Compute local asymmetry for each pixel."""
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY).astype(np.float32)
        h, w = gray.shape
        center_h, center_w = h // 2, w // 2
        
        # Create mirrored versions
        top_half = gray[:center_h, :]
        bottom_half = np.flipud(gray[center_h:, :])
        left_half = gray[:, :center_w]
        right_half = np.fliplr(gray[:, center_w:])
        
        # Vertical asymmetry map
        v_asym = np.zeros_like(gray)
        for i in range(center_h):
            for j in range(w):
                if i < bottom_half.shape[0]:
                    v_asym[i, j] = abs(gray[i, j] - bottom_half[i, j])
                else:
                    v_asym[i, j] = 0.0
        for i in range(center_h, h):
            for j in range(w):
                if i - center_h < top_half.shape[0]:
                    v_asym[i, j] = abs(gray[i, j] - top_half[i - center_h, j])
                else:
                    v_asym[i, j] = 0.0
        
        # Horizontal asymmetry map
        h_asym = np.zeros_like(gray)
        for i in range(h):
            for j in range(center_w):
                if j < right_half.shape[1]:
                    h_asym[i, j] = abs(gray[i, j] - right_half[i, j])
                else:
                    h_asym[i, j] = 0.0
        for i in range(h):
            for j in range(center_w, w):
                if j - center_w < left_half.shape[1]:
                    h_asym[i, j] = abs(gray[i, j] - left_half[i, j - center_w])
                else:
                    h_asym[i, j] = 0.0
        
        # Combine
        asymmetry_map = (v_asym + h_asym) / 2.0
        if asymmetry_map.max() > 0:
            asymmetry_map = asymmetry_map / asymmetry_map.max()
        return asymmetry_map

    def compute_border_irregularity(self, image):
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        return np.sqrt(grad_x**2 + grad_y**2) / 255.0

    def compute_color_variance(self, image):
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
        h_var = ndimage.generic_filter(hsv[:,:,0].astype(float), np.var, size=15)
        s_var = ndimage.generic_filter(hsv[:,:,1].astype(float), np.var, size=15)
        v_var = ndimage.generic_filter(hsv[:,:,2].astype(float), np.var, size=15)
        return (h_var + s_var + v_var) / 3 / 255.0

    def compute_diameter(self, image):
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_OTSU)
        dist = cv2.distanceTransform(binary, cv2.DIST_L2, 5)
        return dist / (dist.max() + 1e-8)

    def compute_mask(self, image):
        if image.dtype != np.uint8:
            image = (np.clip(image, 0, 1) * 255).astype(np.uint8)
        M = (self.w_A * self.compute_asymmetry(image) +
             self.w_B * self.compute_border_irregularity(image) +
             self.w_C * self.compute_color_variance(image) +
             self.w_D * self.compute_diameter(image))
        return (M - M.min()) / (M.max() - M.min() + 1e-8)

# ========================================
# INNOVATION 2: GRADIENT COHERENCE WEIGHTING (GCW) - UNCHANGED
# ========================================
class GradientCoherenceWeighting:
    def __init__(self, model):
        self.model = model

    def extract_feature_gradients(self, input_tensor, target_class=None):
        self.model.eval()
        input_tensor = input_tensor.clone().detach().requires_grad_(True)
        feature_maps = {}
        def hook_fn(module, input, output):
            feature_maps['features'] = output
        handle = None
        for name, module in self.model.backbone.named_modules():
            if 'features.7' in name or 'norm' in name:
                handle = module.register_forward_hook(hook_fn)
                break
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1)
        self.model.zero_grad()
        output[0, target_class].backward()
        gradients = input_tensor.grad.data
        if handle:
            handle.remove()
        return gradients

    def compute_channel_coherence(self, input_tensor):
        gradients = self.extract_feature_gradients(input_tensor)
        grad_mag = torch.abs(gradients).sum(dim=(2, 3))
        channel_norms = torch.norm(grad_mag, p=2, dim=0)
        return channel_norms / (channel_norms.sum() + 1e-8)

    def compute_gcw_attention(self, input_tensor):
        w_c = self.compute_channel_coherence(input_tensor)
        gradients = self.extract_feature_gradients(input_tensor)
        weighted = (torch.abs(gradients) * w_c.view(1, -1, 1, 1)).sum(dim=1, keepdim=True)
        return F.interpolate(weighted, size=(224, 224), mode='bilinear', align_corners=False)

# ========================================
# BASELINE XAI METHODS - FIXED SMOOTHGRAD
# ========================================
class GradCAM:
    def __init__(self, model):
        self.model = model

    def extract_cam(self, input_tensor, target_class=None):
        self.model.eval()
        target_layer = None
        for name, module in self.model.backbone.named_modules():
            if isinstance(module, nn.Conv2d):
                target_layer = module
        if target_layer is None:
            return np.zeros((224, 224))
        
        activations = None
        gradients = None
        
        def forward_hook(module, input, output):
            nonlocal activations
            activations = output.detach()
        
        def backward_hook(module, grad_input, grad_output):
            nonlocal gradients
            gradients = grad_output[0].detach()
        
        forward_handle = target_layer.register_forward_hook(forward_hook)
        backward_handle = target_layer.register_full_backward_hook(backward_hook)
        
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1)
        
        self.model.zero_grad()
        output[0, target_class].backward()
        
        pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])
        for i in range(activations.shape[1]):
            activations[:, i, :, :] *= pooled_gradients[i]
        
        cam = torch.mean(activations, dim=1).squeeze()
        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam = cam.cpu().numpy()
        cam = cv2.resize(cam, (224, 224))
        
        forward_handle.remove()
        backward_handle.remove()
        return cam

class IntegratedGradients:
    def __init__(self, model):
        self.model = model

    def compute_ig(self, input_tensor, target_class=None, steps=30):
        self.model.eval()
        baseline = torch.zeros_like(input_tensor)
        alphas = torch.linspace(0, 1, steps).to(input_tensor.device)
        interpolated_inputs = torch.cat([
            baseline + alpha * (input_tensor - baseline) 
            for alpha in alphas
        ], dim=0)
        interpolated_inputs.requires_grad_(True)
        
        outputs = self.model(interpolated_inputs)
        if target_class is None:
            target_class = outputs[0].argmax()
        
        self.model.zero_grad()
        target_outputs = outputs[:, target_class]
        target_outputs.backward(torch.ones_like(target_outputs))
        
        avg_gradients = torch.mean(interpolated_inputs.grad, dim=0, keepdim=True)
        ig = (input_tensor - baseline) * avg_gradients
        ig = torch.abs(ig).sum(dim=1, keepdim=True)
        ig = F.interpolate(ig, size=(224, 224), mode='bilinear', align_corners=False)
        ig = ig[0, 0].cpu().detach().numpy()
        ig = (ig - ig.min()) / (ig.max() - ig.min() + 1e-8)
        return ig

class SmoothGrad:
    def __init__(self, model, samples=30, noise_level=0.15):
        self.model = model
        self.samples = samples
        self.noise_level = noise_level

    def compute_smoothgrad(self, input_tensor, target_class=None, samples=None, noise_level=None):
        self.model.eval()
        if samples is None:
            samples = self.samples
        if noise_level is None:
            noise_level = self.noise_level
        
        gradients_list = []
        for _ in range(samples):
            noise = torch.randn_like(input_tensor) * noise_level
            noisy_input = input_tensor + noise
            noisy_input.requires_grad_(True)
            
            output = self.model(noisy_input)
            if target_class is None:
                target_class = output.argmax(dim=1)
            
            self.model.zero_grad()
            output[0, target_class].backward()
            gradients_list.append(noisy_input.grad.data)
        
        avg_gradients = torch.mean(torch.stack(gradients_list), dim=0)
        smoothgrad = torch.abs(avg_gradients).sum(dim=1, keepdim=True)
        smoothgrad = F.interpolate(smoothgrad, size=(224, 224), mode='bilinear', align_corners=False)
        smoothgrad = smoothgrad[0, 0].cpu().detach().numpy()
        smoothgrad = (smoothgrad - smoothgrad.min()) / (smoothgrad.max() - smoothgrad.min() + 1e-8)
        return smoothgrad

# ========================================
# COMPREHENSIVE TESTER - FIXED COMBINED VISUALIZATION
# ========================================
class CGHAFTester:
    def __init__(self, model, test_loader, label_map, device):
        self.model = model
        self.test_loader = test_loader
        self.label_map = label_map
        self.device = device
        self.class_names = list(label_map.keys())
        
        self.clinical_mask = ClinicalSaliencyMask()
        self.gcw = GradientCoherenceWeighting(model)
        self.gradcam = GradCAM(model)
        self.integrated_gradients = IntegratedGradients(model)
        self.smoothgrad = SmoothGrad(model)

    def evaluate_model(self):
        print("\n" + "="*60)
        print("MODEL EVALUATION")
        print("="*60)
        self.model.eval()
        all_preds, all_labels, all_probs, all_paths = [], [], [], []
        with torch.no_grad():
            for images, labels, paths in tqdm(self.test_loader, desc="Evaluating"):
                images = images.to(self.device)
                outputs = self.model(images)
                probs = F.softmax(outputs, dim=1)
                preds = outputs.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())
                all_probs.extend(probs.cpu().numpy())
                all_paths.extend(paths)
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        all_probs = np.array(all_probs)
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='weighted')
        print(f"\nAccuracy: {acc:.4f}, F1: {f1:.4f}")
        return all_preds, all_labels, all_probs, all_paths

    def generate_comprehensive_visualizations(self, predictions, labels, paths, images_per_class=5):
        print("\n" + "="*60)
        print("GENERATING CG-HAF XAI VISUALIZATIONS")
        print("="*60)
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        selected_samples = {}
        for class_idx, class_name in enumerate(self.class_names):
            correct = (np.array(labels) == class_idx) & (np.array(predictions) == class_idx)
            indices = np.where(correct)[0]
            sel = np.random.choice(indices, min(images_per_class, len(indices)), replace=False) if len(indices) > 0 else indices
            selected_samples[class_name] = [(paths[i], labels[i]) for i in sel]

        for class_name, samples in selected_samples.items():
            print(f"\nProcessing class: {class_name}")
            class_dir = os.path.join(output_dir, "xai_visualizations", class_name)
            os.makedirs(class_dir, exist_ok=True)
            for i, (img_path, label) in enumerate(samples):
                try:
                    with Image.open(img_path) as img:
                        orig_img = img.convert('RGB').copy()
                    orig_array = np.array(orig_img.resize((224, 224)))
                    img_tensor = transform(orig_img).unsqueeze(0).to(self.device)
                    
                    # Compute maps
                    M_clin = self.clinical_mask.compute_mask(orig_array)
                    A_gcw = self.gcw.compute_gcw_attention(img_tensor)
                    A_gcw_np = A_gcw[0, 0].cpu().detach().numpy()
                    A_gcw_np = (A_gcw_np - A_gcw_np.min()) / (A_gcw_np.max() - A_gcw_np.min() + 1e-8)
                    A_gcw_np = cv2.GaussianBlur(A_gcw_np, (11, 11), 0)
                    
                    gradcam_map = self.gradcam.extract_cam(img_tensor)
                    gradcam_map = cv2.GaussianBlur(gradcam_map, (11, 11), 0)
                    ig_map = self.integrated_gradients.compute_ig(img_tensor, steps=30)
                    ig_map = cv2.GaussianBlur(ig_map, (11, 11), 0)
                    smoothgrad_map = self.smoothgrad.compute_smoothgrad(img_tensor, samples=30)
                    smoothgrad_map = cv2.GaussianBlur(smoothgrad_map, (11, 11), 0)
                    
                    # Save visualizations
                    self._save_innovation1_visualization(orig_array, M_clin, class_name, i+1, class_dir)
                    self._save_innovation2_visualization(orig_array, A_gcw_np, class_name, i+1, class_dir)
                    self._save_combined_visualization(orig_array, M_clin, A_gcw_np, class_name, i+1, class_dir)
                    self._save_baseline_comparison(orig_array, gradcam_map, ig_map, smoothgrad_map, class_name, i+1, class_dir)
                    
                except Exception as e:
                    print(f"Error on {img_path}: {e}")
                clear_memory()

    def _save_innovation1_visualization(self, original_img, M_clin, class_name, sample_idx, save_dir):
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        asymmetry = self.clinical_mask.compute_asymmetry(original_img)
        border = self.clinical_mask.compute_border_irregularity(original_img)
        color_var = self.clinical_mask.compute_color_variance(original_img)
        diameter = self.clinical_mask.compute_diameter(original_img)
        
        axes[0, 0].imshow(original_img); axes[0, 0].set_title('Original'); axes[0, 0].axis('off')
        im1 = axes[0, 1].imshow(asymmetry, cmap='hot'); axes[0, 1].set_title('A: Asymmetry'); axes[0, 1].axis('off'); plt.colorbar(im1, ax=axes[0,1])
        im2 = axes[0, 2].imshow(border, cmap='hot'); axes[0, 2].set_title('B: Border'); axes[0, 2].axis('off'); plt.colorbar(im2, ax=axes[0,2])
        im3 = axes[1, 0].imshow(color_var, cmap='hot'); axes[1, 0].set_title('C: Color Var'); axes[1, 0].axis('off'); plt.colorbar(im3, ax=axes[1,0])
        im4 = axes[1, 1].imshow(diameter, cmap='hot'); axes[1, 1].set_title('D: Diameter'); axes[1, 1].axis('off'); plt.colorbar(im4, ax=axes[1,1])
        im5 = axes[1, 2].imshow(M_clin, cmap='hot'); axes[1, 2].set_title('Combined ABCD'); axes[1, 2].axis('off'); plt.colorbar(im5, ax=axes[1,2])
        
        plt.suptitle(f'Innovation 1: Clinical Saliency Mask (ABCD)\n{class_name} - Sample {sample_idx}')
        plt.savefig(os.path.join(save_dir, f"{class_name}_sample{sample_idx}_innovation1.png"), dpi=300, bbox_inches='tight')
        plt.close()

    def _save_innovation2_visualization(self, original_img, A_gcw, class_name, sample_idx, save_dir):
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(original_img); axes[0].set_title('Original'); axes[0].axis('off')
        im = axes[1].imshow(A_gcw, cmap='hot'); axes[1].set_title('GCW Map'); axes[1].axis('off'); plt.colorbar(im, ax=axes[1])
        axes[2].imshow(original_img); axes[2].imshow(A_gcw, cmap='hot', alpha=0.5); axes[2].set_title('GCW Overlay'); axes[2].axis('off')
        plt.suptitle(f'Innovation 2: Gradient Coherence Weighting\n{class_name} - Sample {sample_idx}')
        plt.savefig(os.path.join(save_dir, f"{class_name}_sample{sample_idx}_innovation2.png"), dpi=300, bbox_inches='tight')
        plt.close()

    def _save_combined_visualization(self, original_img, M_clin, A_gcw, class_name, sample_idx, save_dir):
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(original_img); axes[0].set_title('Original'); axes[0].axis('off')
        
        # Normalize both maps before combination
        M_clin_norm = (M_clin - M_clin.min()) / (M_clin.max() - M_clin.min() + 1e-8)
        A_gcw_norm = (A_gcw - A_gcw.min()) / (A_gcw.max() - A_gcw.min() + 1e-8)
        # Weighted fusion: clinical mask has higher weight
        fusion = 0.7 * M_clin_norm + 0.3 * A_gcw_norm
        fusion = cv2.GaussianBlur(fusion, (11, 11), 0)
        
        im = axes[1].imshow(fusion, cmap='plasma'); axes[1].set_title('CG-HAF Combined'); axes[1].axis('off'); plt.colorbar(im, ax=axes[1])
        axes[2].imshow(original_img); axes[2].imshow(fusion, cmap='plasma', alpha=0.6); axes[2].set_title('Combined Overlay'); axes[2].axis('off')
        plt.suptitle(f'CG-HAF Combined: Clinical + GCW\n{class_name} - Sample {sample_idx}')
        plt.savefig(os.path.join(save_dir, f"{class_name}_sample{sample_idx}_combined.png"), dpi=300, bbox_inches='tight')
        plt.close()

    def _save_baseline_comparison(self, original_img, gradcam, ig, smoothgrad, class_name, sample_idx, save_dir):
        fig, axes = plt.subplots(2, 2, figsize=(12, 12))
        axes[0, 0].imshow(original_img); axes[0, 0].set_title('Original'); axes[0, 0].axis('off')
        axes[0, 1].imshow(gradcam, cmap='jet'); axes[0, 1].set_title('Grad-CAM'); axes[0, 1].axis('off')
        axes[1, 0].imshow(ig, cmap='viridis'); axes[1, 0].set_title('Integrated Gradients'); axes[1, 0].axis('off')
        axes[1, 1].imshow(smoothgrad, cmap='plasma'); axes[1, 1].set_title('SmoothGrad'); axes[1, 1].axis('off')
        plt.suptitle(f'Baseline XAI Methods\n{class_name} - Sample {sample_idx}')
        plt.savefig(os.path.join(save_dir, f"{class_name}_sample{sample_idx}_baselines.png"), dpi=300, bbox_inches='tight')
        plt.close()

# ========================================
# MAIN PIPELINE
# ========================================
def main():
    print("="*60)
    print("CG-HAF: CLINICALLY-GUIDED XAI FOR DERMATOLOGY")
    print("FINAL CORRECTED VERSION")
    print("="*60)

    MODEL_PATH = "/kaggle/input/bestu/pytorch/default/1/swin_vit_xai_model.pth"
    METADATA_PATH = "/kaggle/input/testing-set/ham10000_test_dataset_4500.csv"
    IMG_DIR1 = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1"
    IMG_DIR2 = "/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2"

    checkpoint = torch.load(MODEL_PATH, map_location=device)
    label_map = checkpoint['label_map']
    use_bn = any('running_mean' in k for k in checkpoint['model_state_dict'].keys())
    
    model = EnhancedSwinTransformer(len(label_map), use_batchnorm=use_bn).to(device)
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    model.eval()

    df = load_test_data(METADATA_PATH, IMG_DIR1, IMG_DIR2)
    df['label'] = df['dx'].map(label_map)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    test_dataset = HAM10000Dataset(df, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

    tester = CGHAFTester(model, test_loader, label_map, device)
    preds, labels, probs, paths = tester.evaluate_model()
    tester.generate_comprehensive_visualizations(preds, labels, paths, images_per_class=5)

    print("\n" + "="*60)
    print("CG-HAF TESTING COMPLETED SUCCESSFULLY!")
    print("✅ Innovation 1: CSM (Fixed Asymmetry)")
    print("✅ Innovation 2: GCW (Perfect)")
    print("✅ Combined Visualization (No Pink Hue)")
    print("="*60)

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        clear_memory()